# Colab 34 — Objective **and** loss weighting, band-decomposed (+ protocol audit)

**Two questions, one controlled harness.**

**Q-A — Is the objective the thing that costs rank fidelity?**
`WRITEUP.md` (colab32 2x2) says the objective is near-neutral once pooling is present: reg->clf Spearman
deltas of about `-0.00 / -0.05 / -0.01 / -0.09` on synth / 3Di / SS / AA. But colab33 seed 0 reports
**3Di Spearman 0.33** for `reg-pool`, where the deck (colab29b, `clf-pool`) shows **0.93**. Those cannot
both describe the same effect.

**The likely resolution is that this is not a reg-vs-clf gap at all.** colab32 and colab33 use *identical*
band weights (`0.5 / 2.0 / 4.0`) and *identical* protocol constants (`N_TRAIN=30_000`, `SEEDS=[0,1,2]`,
`EPOCHS=30`, `STRAT_PER_BIN=400`, `STRAT_CAND=200_000`, `SYN_PERTURB/INDEP=20_000/8_000`, synth-feed seed
`20260810`). So colab32's `reg-pool` **is** colab33's SNNEED. If colab32's clf-minus-reg gap on 3Di is
only `-0.05`, then colab32's `clf-pool` also sits near 0.3 on 3Di, not at the deck's 0.93 — which makes
the discrepancy **colab32/33 vs colab29b (protocol)**, affecting *both* objectives equally.

There is already hard evidence of protocol divergence: `colab33_metrics.csv` has **blank AUROC and MAP@10
for the AA feed** (the signature of an AA oracle with zero pairs at `normLev >= 0.70`), while `WRITEUP.md`
reports colab32 AA MAP@10 = 0.942 and 3Di MAP@10 = 0.500 against colab33's 0.740. Same constants,
different pools.

> **Therefore section 3 prints a POOL/ORACLE AUDIT before any training.** If the audit disagrees with
> colab32/colab29b, the objective comparison is not the story and the pool construction is.

**Q-B — Is the *weighting*, not the objective, what costs rank fidelity?**
Band-weighted MSE tells the encoder to care 8x less about pairs below 0.30 (`w=0.5` vs `w=4.0`).
Cross-entropy has no such tilt: it must separate far|mid *and* mid|high, so it is **uniformly attentive
across the range**. Feeds whose mass sits in the down-weighted band (AA median 0.20, 3Di median 0.24)
would be the ones to suffer. That is a hyperparameter, not an architecture — so it is worth one arm.

`W_FAR/W_MID/W_HIGH = 0.5/2.0/4.0` has been carried unchanged since **colab14**, where it was introduced,
and colab14's own notes already flagged it as possibly too aggressive:

> "*AUROC near 0.5 — the band-weighting was too aggressive and destabilized training (try `W_HIGH=2,
> W_MID=1.5, W_FAR=0.8` as a softer alternative).*"

That softer setting has **never been tested**. It is arm 4 here.

---

### The four arms (pooling fixed ON in all of them; only the loss differs)

| Arm | Objective | Weights (far / mid / high) | Provenance |
|---|---|---|---|
| `clf-pool`  | 3-bin cross-entropy | n/a (uniform over bins) | colab16 -> deck model |
| `reg-band`  | MSE on `normLev`    | **0.5 / 2.0 / 4.0** | colab14 -> colab33 deployed |
| `reg-flat`  | MSE on `normLev`    | **1.0 / 1.0 / 1.0** | never tested |
| `reg-soft`  | MSE on `normLev`    | **0.8 / 1.5 / 2.0** | colab14's own suggested fix, never tested |

### What is measured

- **Geometry metrics are computed identically for every arm, from the encoder cosine only** — that is
  what makes the arms comparable, and it is what retrieval actually uses. Heads are never used for these.
- **Spearman is decomposed by band** (`< 0.30`, `[0.30, 0.70)`, `>= 0.70`) as well as overall. This is
  the measurement that decides Q-B: if band weighting is the cost, the deficit is concentrated below 0.30.
- **Value fidelity (RMSE on `>= 0.70`) uses each arm's own native readout** — `1 - ||d||/2` for the
  regression arms, `E[bin midpoint]` for the classifier. This is the one place the head is used.

### What this notebook does NOT change
Encoder, pooling (`K=16`), training size (30k), epochs (30), seeds, pools, oracles and stratified pair
construction are all held identical to colab32/colab33 so the numbers are drop-in comparable.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f); print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy matplotlib --quiet

### 1a. Version capture (Methods chapter dependency)

`requirements.txt` pins `torch`, `numpy` and `matplotlib` but **not** `rapidfuzz`, `scikit-learn` or
`scipy`, and Colab resolves them at install time — so the versions that actually produced every number in
this thesis are currently unrecorded. This cell writes `environment_colab34.json` next to the results.
Commit it with the CSV; the Methods chapter cites it instead of guessing.

In [ ]:
import json, platform, subprocess, sys, datetime

def _v(mod):
    try:
        return __import__(mod).__version__
    except Exception as e:
        return f'<unavailable: {e}>'

ENV = {
    'captured_utc':   datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    'python':         sys.version.split()[0],
    'platform':       platform.platform(),
    'torch':          _v('torch'),
    'numpy':          _v('numpy'),
    'pandas':         _v('pandas'),
    'scipy':          _v('scipy'),
    'sklearn':        _v('sklearn'),
    'rapidfuzz':      _v('rapidfuzz'),
    'matplotlib':     _v('matplotlib'),
}
try:
    import torch as _t
    ENV['cuda_available'] = _t.cuda.is_available()
    ENV['cuda_device']    = _t.cuda.get_device_name(0) if _t.cuda.is_available() else None
    ENV['cuda_version']   = _t.version.cuda
except Exception:
    pass
try:
    ENV['git_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
except Exception:
    ENV['git_commit'] = '<not a git checkout>'

# NOTE FOR METHODS: the CATH release/version is NOT recorded anywhere in the repo.
# Fill this in by hand once confirmed, then keep it here.
ENV['cath_release'] = 'TODO — record exact CATH release, S20 file name, and download date'
ENV['3di_source']   = 'TODO — record Foldseek version used to generate 3Di strings'

with open('environment_colab34.json', 'w') as fh:
    json.dump(ENV, fh, indent=2)
for k, v in ENV.items():
    print(f'{k:<16} {v}')

In [ ]:
import time, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy import sparse
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# --- held identical to colab32 / colab33 so results are drop-in comparable ---
QUICK   = False                       # True -> SEEDS=[0] for a fast smoke pass
N_TRAIN = 30_000
SEEDS   = [0] if QUICK else [0, 1, 2]
EPOCHS  = 30
STRAT_PER_BIN, STRAT_CAND = 400, 200_000
SYN_PERTURB, SYN_INDEP    = 20_000, 8_000
FEED_ORDER = ['synth', '3Di', 'SS', 'AA']
print(f'seeds={SEEDS}  epochs={EPOCHS}  n_train={N_TRAIN}')

## 2. Constants, encoder, and the four arms

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
MIN_LEN, MAX_LEN, BS, K = 50, 200, 128, 16
BAND_LOW_AA, BAND_HIGH = 0.30, 0.70
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)

def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

def encode_pad(seq):
    idx = [CHAR_TO_IDX[c] for c in seq][:MAX_LEN]; idx += [PAD_IDX]*(MAX_LEN-len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, abc, rng):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= MAX_LEN: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def rand_seq(abc, rng):
    L = int(rng.integers(MIN_LEN, MAX_LEN+1)); return ''.join(rng.choice(list(abc), size=L))

def bin_idx(x): return 0 if x < BAND_LOW_AA else (1 if x < BAND_HIGH else 2)
BIN_MID = np.array([BAND_LOW_AA/2, (BAND_LOW_AA+BAND_HIGH)/2, (BAND_HIGH+1.0)/2], dtype=np.float32)
print('classifier value readout midpoints (far/mid/high):', BIN_MID)

In [ ]:
class EncPool(nn.Module):
    # Identical to colab32/colab33: emb -> 2x Conv1d -> AdaptiveAvgPool1d(K) -> fc -> L2 normalise.
    def __init__(s):
        super().__init__(); s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1 = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, 64, 3, padding=1)
        s.pool = nn.AdaptiveAvgPool1d(K); s.fc = nn.Linear(64*K, 128)
    def forward(s, x):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        return F.normalize(s.fc(s.pool(h).flatten(1)), p=2, dim=1)

class RegModel(nn.Module):
    # Parameter-free readout: no head to train, no head to discard.
    def __init__(s, enc): super().__init__(); s.encoder = enc
    def forward(s, a, b):
        ea, eb = s.encoder(a), s.encoder(b)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0

class ClfModel(nn.Module):
    # colab16 head: Linear(128->64) -> LeakyReLU -> Linear(64->3) on |e_a - e_b|.
    def __init__(s, enc):
        super().__init__(); s.encoder = enc
        s.head = nn.Sequential(nn.Linear(128, 64), nn.LeakyReLU(), nn.Linear(64, 3))
    def forward(s, a, b):
        return s.head(torch.abs(s.encoder(a) - s.encoder(b)))

def make_w(w_far, w_mid, w_high):
    def f(y):
        w = torch.full_like(y, w_mid)
        w[y < BAND_LOW_AA] = w_far; w[y >= BAND_HIGH] = w_high
        return w
    return f

ARMS = {
    'clf-pool': dict(obj='clf', w=None,                    note='colab16 -> deck model'),
    'reg-band': dict(obj='reg', w=make_w(0.5, 2.0, 4.0),   note='colab14 -> colab33 deployed'),
    'reg-flat': dict(obj='reg', w=make_w(1.0, 1.0, 1.0),   note='unweighted — never tested'),
    'reg-soft': dict(obj='reg', w=make_w(0.8, 1.5, 2.0),   note="colab14's own suggested fix"),
}
ARM_ORDER = list(ARMS)
for a, c in ARMS.items(): print(f'  {a:<9} obj={c["obj"]:<3}  {c["note"]}')

In [ ]:
class PairDS(Dataset):
    def __init__(s, pp, obj): s.p, s.obj = pp, obj
    def __len__(s): return len(s.p)
    def __getitem__(s, i):
        a, b, l = s.p[i]
        y = torch.tensor(bin_idx(l)) if s.obj == 'clf' else torch.tensor(l, dtype=torch.float32)
        return encode_pad(a), encode_pad(b), y

def build_pairs(n, seed):
    # Target-uniform synthetic AA pairs - identical generator to colab32/colab33.
    rng = np.random.default_rng(seed); pairs = []
    while len(pairs) < n:
        sd = rand_seq(AA_ALPHABET, rng); L = len(sd)
        t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, AA_ALPHABET, rng)
        if 1 <= len(o) <= MAX_LEN: pairs.append((sd, o, norm_lev(sd, o)))
    return pairs

def train_arm(arm, pairs, seed):
    cfg = ARMS[arm]; torch.manual_seed(seed)
    enc = EncPool()
    model = (ClfModel(enc) if cfg['obj'] == 'clf' else RegModel(enc)).to(device)
    dl  = DataLoader(PairDS(pairs, cfg['obj']), batch_size=BS, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), 1e-3)
    crit = nn.CrossEntropyLoss() if cfg['obj'] == 'clf' else None
    model.train(); t0 = time.time()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for a, b, y in dl:
            a, b, y = a.to(device), b.to(device), y.to(device)
            if cfg['obj'] == 'clf':
                loss = crit(model(a, b), y)
            else:
                loss = (cfg['w'](y) * (model(a, b) - y)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1:
            print(f'    [{arm} s={seed}] epoch {ep:>2}/{EPOCHS}  loss {tot/nb:.4f}')
    if device.type == 'cuda': torch.cuda.synchronize()
    model.eval()
    print(f'    [{arm} s={seed}] trained in {time.time()-t0:.0f}s')
    return model

## 3. Pools, oracles, stratified pairs — **and the protocol audit**

Code below is lifted unchanged from colab33 so any divergence is attributable to the data, not to a
rewrite. The audit block at the end is the new part: it prints exactly the quantities that colab32,
colab33 and colab29b disagree about.

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

# NOTE FOR METHODS: these two domains were added after observing that they create high-similarity AA
# pairs. That is an outcome-aware filter and cannot be written as a generic rule in a Methods chapter.
# Flagged in METHODS_OUTLINE.md 3.4 — resolve before the chapter is frozen. Kept here only so this
# notebook reproduces colab33's pool exactly.
RESCUED = {'4z0mC02', '3qkaE02'}

def _valid(seq, isstd, d):
    return (isinstance(seq, str) and isstd(seq) and ((MIN_LEN <= len(seq) <= MAX_LEN) or d in RESCUED))
id_to_aa  = {d: s for d, s in zip(raw['domain_id'], raw['aa_seq'])              if _valid(s, is_aa, d)}
id_to_ss  = {d: s for d, s in zip(raw['domain_id'], raw['ss_seq'])              if _valid(s, is_ss, d)}
id_to_3di = {d: s for d, s in zip(seqs3['domain_id'], seqs3['3di'].astype(str)) if _valid(s, is_aa, d)}
LOOK = {'AA': id_to_aa, 'SS': id_to_ss, '3Di': id_to_3di}
POOL_SEQ = {f: list(LOOK[f].values()) for f in LOOK}
CATH_FEEDS = ['AA', 'SS', '3Di']
for f in CATH_FEEDS: print(f'  {f:<4} pool = {len(POOL_SEQ[f]):>6}')

In [ ]:
def build_oracle(feed, block=1024):
    seqs = POOL_SEQ[feed]; lens = np.array([len(s) for s in seqs]); N = len(seqs)
    T_high = {}; pos = []
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a].copy(); row[i] = -1.0
            hi = np.where(row >= BAND_HIGH)[0]
            if hi.size: T_high[i] = hi.astype(np.int32)
            for j in hi:
                if j > i: pos.append((i, int(j), float(row[j])))
    return dict(T_high=T_high, pos_pairs=pos)

ORACLE = {}
for f in CATH_FEEDS:
    t0 = time.time(); print(f'oracle {f} (SS is the slow one)...')
    ORACLE[f] = build_oracle(f)
    print(f'  {f}: queries@0.70={len(ORACLE[f]["T_high"])}, pos pairs={len(ORACLE[f]["pos_pairs"])}  [{time.time()-t0:.0f}s]')

In [ ]:
def build_strat_pairs(feed, rng):
    seqs = POOL_SEQ[feed]; N = len(seqs)
    a = rng.integers(0, N, STRAT_CAND); b = rng.integers(0, N, STRAT_CAND)
    keep = a != b; a, b = a[keep], b[keep]
    nl = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    if ORACLE[feed]['pos_pairs']:
        pa = np.array(ORACLE[feed]['pos_pairs'], float)
        a  = np.concatenate([a, pa[:, 0].astype(np.int64)])
        b  = np.concatenate([b, pa[:, 1].astype(np.int64)])
        nl = np.concatenate([nl, pa[:, 2]])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9); ai, aj, av = [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size:
            t = rng.permutation(idx)[:STRAT_PER_BIN]; ai.append(a[t]); aj.append(b[t]); av.append(nl[t])
    return dict(i=np.concatenate(ai).astype(np.int64),
                j=np.concatenate(aj).astype(np.int64),
                nl=np.concatenate(av))
STRAT = {f: build_strat_pairs(f, np.random.default_rng(999)) for f in CATH_FEEDS}

def build_synth_feed(n_perturb, n_indep, per_bin=STRAT_PER_BIN, seed=20260810):
    r = np.random.default_rng(seed); recs = []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r)
        part = perturb(base, int(r.integers(0, len(base)+1)), AA_ALPHABET, r)
        if 1 <= len(part) <= MAX_LEN: recs.append((base, part))
    for _ in range(n_indep): recs.append((rand_seq(AA_ALPHABET, r), rand_seq(AA_ALPHABET, r)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]; nl = np.array([x[2] for x in recs])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9); take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: take.extend(r.permutation(idx)[:per_bin].tolist())
    seqs, I, J, NL = [], [], [], []
    for idx in take:
        a, b, l = recs[int(idx)]
        I.append(len(seqs)); seqs.append(a); J.append(len(seqs)); seqs.append(b); NL.append(l)
    return seqs, np.array(I), np.array(J), np.array(NL)

SYN_SEQ, SYN_I, SYN_J, SYN_NL = build_synth_feed(SYN_PERTURB, SYN_INDEP)
POOL_SEQ['synth'] = SYN_SEQ; ORACLE['synth'] = build_oracle('synth')
print(f'synth: pool={len(SYN_SEQ)}, queries@0.70={len(ORACLE["synth"]["T_high"])}')

In [ ]:
# ============================== PROTOCOL AUDIT ==============================
# Everything colab32 / colab33 / colab29b disagree about, printed before any training.
def _pairs(feed):
    if feed == 'synth': return SYN_I, SYN_J, SYN_NL
    P = STRAT[feed]; return P['i'], P['j'], P['nl']

print(f'{"feed":<7}{"pool":>7}{"queries@.70":>13}{"oracle pos":>12}'
      f'{"strat n":>10}{"strat>=.70":>12}{"strat<.30":>11}{"strat mid":>11}')
print('-'*83)
AUDIT = {}
for f in FEED_ORDER:
    I, J, nl = _pairs(f)
    row = dict(pool=len(POOL_SEQ[f]),
               queries_at_070=len(ORACLE[f]['T_high']),
               oracle_pos_pairs=len(ORACLE[f]['pos_pairs']),
               strat_n=int(len(nl)),
               strat_high=int((nl >= BAND_HIGH).sum()),
               strat_far=int((nl < BAND_LOW_AA).sum()),
               strat_mid=int(((nl >= BAND_LOW_AA) & (nl < BAND_HIGH)).sum()),
               strat_median=float(np.median(nl)))
    AUDIT[f] = row
    print(f'{f:<7}{row["pool"]:>7}{row["queries_at_070"]:>13}{row["oracle_pos_pairs"]:>12}'
          f'{row["strat_n"]:>10}{row["strat_high"]:>12}{row["strat_far"]:>11}{row["strat_mid"]:>11}')

print('\nEXPECTED, from the record:')
print('  colab32 (WRITEUP.md): AA MAP@10 = 0.942 and 3Di MAP@10 = 0.500 -> AA oracle NOT empty')
print('  colab33 (metrics.csv): AA AUROC and MAP@10 both blank        -> AA oracle IS empty')
print('  deck colab29b:         3Di Spearman 0.93 vs colab33 0.33')
print()
for f in FEED_ORDER:
    if AUDIT[f]['queries_at_070'] == 0:
        print(f'  *** WARNING: {f} has ZERO queries at >=0.70 — AUROC and MAP@10 will be NaN for {f}.')
    if AUDIT[f]['strat_high'] == 0:
        print(f'  *** WARNING: {f} stratified set contains NO pair >=0.70 — AUROC undefined, '
              f'high-band Spearman and RMSE will be NaN.')
    if 0 < AUDIT[f]['queries_at_070'] <= 20:
        print(f'  *** CAUTION: {f} has only {AUDIT[f]["queries_at_070"]} queries at >=0.70 — '
              f'MAP@10/AUROC for {f} are effectively anecdotes, report n alongside.')
with open('colab34_audit.json', 'w') as fh: json.dump(AUDIT, fh, indent=2)

## 4. Metrics

**Geometry (comparable across arms) uses the encoder cosine only** — never a head. Spearman is reported
overall *and* decomposed into the three bands, because that decomposition is what separates
"the objective costs ranking" from "the band weighting costs ranking".

**Value fidelity uses each arm's native readout**, which is the only asymmetric measurement here and the
only place the classifier head is touched.

In [ ]:
def _auroc(sim, nl):
    y = (nl >= BAND_HIGH).astype(int)
    return roc_auc_score(y, sim) if 0 < y.sum() < len(y) else np.nan

def map10_emb(E_t, T_high, k=10, qb=256):
    q = list(T_high.keys())
    if not q: return np.nan
    aps = []
    for s0 in range(0, len(q), qb):
        qi = q[s0:s0+qb]; sc = E_t[qi] @ E_t.t()
        for r, idx in enumerate(qi): sc[r, idx] = -1e9
        top = torch.topk(sc, k, dim=1).indices.cpu().numpy()
        for r, idx in enumerate(qi):
            ts = set(T_high[idx].tolist()); hits = 0; ap = 0.0
            for rr, o in enumerate(top[r], 1):
                if o in ts: hits += 1; ap += hits / rr
            aps.append(ap / min(len(ts), k))
    return float(np.mean(aps))

BANDS = {'far': lambda nl: nl < BAND_LOW_AA,
         'mid': lambda nl: (nl >= BAND_LOW_AA) & (nl < BAND_HIGH),
         'high': lambda nl: nl >= BAND_HIGH}

def _rho(sim, nl):
    if len(nl) < 10 or np.ptp(nl) == 0: return np.nan
    r = spearmanr(sim, nl).correlation
    return float(r) if r == r else np.nan

@torch.no_grad()
def snn_embed(model, feed, bs=256):
    seqs = POOL_SEQ[feed]; out = []
    for i in range(0, len(seqs), bs):
        x = torch.stack([encode_pad(s) for s in seqs[i:i+bs]]).to(device)
        out.append(model.encoder(x).cpu().numpy())
    return np.concatenate(out).astype(np.float32)

@torch.no_grad()
def native_pred(model, arm, feed, E_np, chunk=4096):
    # Each arm's own value readout on the stratified pairs - for RMSE only.
    I, J, _ = _pairs(feed)
    if ARMS[arm]['obj'] == 'reg':
        return 1.0 - np.linalg.norm(E_np[I] - E_np[J], axis=1) / 2.0
    out = []
    for s0 in range(0, len(I), chunk):
        ea = torch.as_tensor(E_np[I[s0:s0+chunk]], device=device)
        eb = torch.as_tensor(E_np[J[s0:s0+chunk]], device=device)
        p = torch.softmax(model.head(torch.abs(ea - eb)), dim=1).cpu().numpy()
        out.append(p @ BIN_MID)
    return np.concatenate(out)

def eval_arm(model, arm, feed):
    E = snn_embed(model, feed)
    I, J, nl = _pairs(feed)
    sim = np.sum(E[I] * E[J], axis=1)                     # cosine — comparable across arms
    rec = dict(arm=arm, feed=feed,
               spearman=_rho(sim, nl),
               auroc=_auroc(sim, nl),
               map10=map10_emb(torch.as_tensor(E, device=device), ORACLE[feed]['T_high']),
               n_pairs=int(len(nl)), n_queries=len(ORACLE[feed]['T_high']))
    for bname, bfn in BANDS.items():                       # band-decomposed rank fidelity
        m = bfn(nl)
        rec[f'spearman_{bname}'] = _rho(sim[m], nl[m])
        rec[f'n_{bname}'] = int(m.sum())
    pv = native_pred(model, arm, feed, E)                  # value fidelity, native readout
    hm = nl >= BAND_HIGH
    rec['rmse_high'] = float(np.sqrt(np.mean((pv[hm] - nl[hm])**2))) if hm.sum() else np.nan
    rec['rmse_all']  = float(np.sqrt(np.mean((pv - nl)**2)))
    return rec

## 5. Train and evaluate — 4 arms x seeds

All four arms in a seed share **the same 30k training pairs**, so within a seed the only difference is
the loss. Expect roughly 1-2 min per arm per seed on a GPU.

In [ ]:
rows = []
for seed in SEEDS:
    print(f'\n=========== seed {seed} ===========')
    pairs = build_pairs(N_TRAIN, seed)      # shared by all four arms this seed
    lab = np.array([l for *_, l in pairs])
    print(f'  training pairs: n={len(pairs)}  median normLev={np.median(lab):.3f}  '
          f'far={int((lab<BAND_LOW_AA).sum())} mid={int(((lab>=BAND_LOW_AA)&(lab<BAND_HIGH)).sum())} '
          f'high={int((lab>=BAND_HIGH).sum())}')
    for arm in ARM_ORDER:
        model = train_arm(arm, pairs, seed)
        for feed in FEED_ORDER:
            r = eval_arm(model, arm, feed); r['seed'] = seed; rows.append(r)
        got = {r['feed']: r['spearman'] for r in rows[-len(FEED_ORDER):]}
        print(f'    -> {arm}: ' + '  '.join(f'{f}:rho={got[f]:.2f}' for f in FEED_ORDER))
        del model
        if device.type == 'cuda': torch.cuda.empty_cache()

df = pd.DataFrame(rows)
df.to_csv('colab34_metrics.csv', index=False)
print(f'\nsaved colab34_metrics.csv  ({len(df)} rows)')

## 6. Results

In [ ]:
agg = (df.groupby(['arm', 'feed'])
         .agg(spearman=('spearman','mean'), sp_far=('spearman_far','mean'),
              sp_mid=('spearman_mid','mean'), sp_high=('spearman_high','mean'),
              auroc=('auroc','mean'), map10=('map10','mean'),
              rmse_high=('rmse_high','mean'), n_high=('n_high','first'),
              n_queries=('n_queries','first'))
         .reset_index())
agg['arm']  = pd.Categorical(agg['arm'], ARM_ORDER, ordered=True)
agg['feed'] = pd.Categorical(agg['feed'], FEED_ORDER, ordered=True)
agg = agg.sort_values(['feed', 'arm'])
pd.set_option('display.width', 200, 'display.max_columns', 40)
print(agg.to_string(index=False, float_format=lambda x: f'{x:6.3f}'))

In [ ]:
print('Q-A — objective effect, holding weights at the deployed 0.5/2/4 (clf-pool minus reg-band):\n')
piv = agg.pivot(index='feed', columns='arm', values='spearman')
for f in FEED_ORDER:
    d = piv.loc[f, 'clf-pool'] - piv.loc[f, 'reg-band']
    print(f'  {f:<6} clf {piv.loc[f,"clf-pool"]:+.3f}   reg-band {piv.loc[f,"reg-band"]:+.3f}   delta {d:+.3f}')
print('\n  colab32 reported deltas: synth -0.00 | 3Di -0.05 | SS -0.01 | AA -0.09')

print('\n\nQ-B — weighting effect within the regression objective (reg-flat minus reg-band):\n')
for f in FEED_ORDER:
    d = piv.loc[f, 'reg-flat'] - piv.loc[f, 'reg-band']
    print(f'  {f:<6} reg-flat {piv.loc[f,"reg-flat"]:+.3f}   reg-band {piv.loc[f,"reg-band"]:+.3f}   delta {d:+.3f}')

print('\n\nWhere is the deficit? Band-decomposed Spearman (the deciding measurement):\n')
print(agg[['feed','arm','sp_far','sp_mid','sp_high','n_high']]
        .to_string(index=False, float_format=lambda x: f'{x:6.3f}'))

print('\n\nValue fidelity — RMSE on pairs >=0.70, each arm using its NATIVE readout (lower is better):\n')
print(agg.pivot(index='feed', columns='arm', values='rmse_high')
         .to_string(float_format=lambda x: f'{x:6.3f}'))
print('\n  WRITEUP.md (colab32) reference: reg-pool beats clf-pool on every feed;')
print('  SS 0.061 vs 0.123, 3Di 0.075 vs 0.096, synth 0.108 vs 0.122, AA 0.115 vs 0.135.')

In [ ]:
import matplotlib.pyplot as plt
FEED_C = {'synth': '#E8871A', '3Di': '#2E6DB4', 'SS': '#C0392B', 'AA': '#8A8F98'}
x = np.arange(len(ARM_ORDER)); w = 0.2

fig, ax = plt.subplots(2, 2, figsize=(15, 9))

for k, f in enumerate(FEED_ORDER):
    v = [agg[(agg.arm==a)&(agg.feed==f)]['spearman'].values[0] for a in ARM_ORDER]
    ax[0,0].bar(x + (k-1.5)*w, v, w, label=f, color=FEED_C[f])
ax[0,0].set_title('Spearman rho (overall) — encoder cosine'); ax[0,0].set_xticks(x)
ax[0,0].set_xticklabels(ARM_ORDER); ax[0,0].axhline(0, color='k', lw=.8); ax[0,0].legend(fontsize=8)

bandcols = ['sp_far', 'sp_mid', 'sp_high']
for k, f in enumerate(FEED_ORDER):
    for bi, bc in enumerate(bandcols):
        v = [agg[(agg.arm==a)&(agg.feed==f)][bc].values[0] for a in ARM_ORDER]
        ax[0,1].plot(x, v, marker='os^'[bi], ls=['-','--',':'][bi], color=FEED_C[f],
                     label=f'{f} {bc[3:]}' if k < 4 else None, alpha=.85)
ax[0,1].set_title('Spearman rho decomposed by band  (o=far  s=mid  ^=high)')
ax[0,1].set_xticks(x); ax[0,1].set_xticklabels(ARM_ORDER); ax[0,1].axhline(0, color='k', lw=.8)
ax[0,1].legend(fontsize=6, ncol=4)

for k, f in enumerate(FEED_ORDER):
    v = [agg[(agg.arm==a)&(agg.feed==f)]['map10'].values[0] for a in ARM_ORDER]
    ax[1,0].bar(x + (k-1.5)*w, v, w, label=f, color=FEED_C[f])
ax[1,0].set_title('MAP@10 (full-pool retrieval)'); ax[1,0].set_xticks(x)
ax[1,0].set_xticklabels(ARM_ORDER); ax[1,0].legend(fontsize=8)

for k, f in enumerate(FEED_ORDER):
    v = [agg[(agg.arm==a)&(agg.feed==f)]['rmse_high'].values[0] for a in ARM_ORDER]
    ax[1,1].bar(x + (k-1.5)*w, v, w, label=f, color=FEED_C[f])
ax[1,1].set_title('RMSE on >=0.70, native readout (lower better)'); ax[1,1].set_xticks(x)
ax[1,1].set_xticklabels(ARM_ORDER); ax[1,1].legend(fontsize=8)

plt.suptitle(f'colab34 — objective and loss weighting, pooling fixed ON  (seeds={SEEDS}, N={N_TRAIN})')
plt.tight_layout(); plt.savefig('colab34_objective_weighting.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
agg.to_csv('colab34_summary.csv', index=False)
print('written:')
for f in ['colab34_metrics.csv', 'colab34_summary.csv', 'colab34_audit.json',
          'colab34_objective_weighting.png', 'environment_colab34.json']:
    print(f'  {f}  ({os.path.getsize(f)} bytes)' if os.path.exists(f) else f'  MISSING {f}')
try:                                    # the Colab VM is ephemeral - keep the artefacts
    from google.colab import files
    for f in ['colab34_metrics.csv', 'colab34_summary.csv', 'colab34_audit.json',
              'colab34_objective_weighting.png', 'environment_colab34.json']:
        if os.path.exists(f): files.download(f)
except Exception as e:
    print('not on Colab / download skipped:', e)

## 7. How to read the result

**Read the audit table first.** If `queries@0.70` or `strat>=0.70` for AA is 0, then AA's AUROC / MAP@10 /
high-band Spearman / RMSE are all NaN by construction, and no arm comparison on AA means anything. That
alone would explain colab33's blank AA columns and would make the pool — not the objective — the finding.

Then, in order:

1. **If `clf-pool` and `reg-band` are close on 3Di (delta near colab32's -0.05) and both sit far below the
   deck's 0.93** -> the discrepancy is **protocol** (colab32/33 vs colab29b), not the objective. The pivot
   to regression is unaffected; what needs auditing is why colab29b's 3Di stratified set scores so much
   higher. Next step: diff colab29b's pool/oracle construction against the audit table above.
2. **If `reg-flat` (or `reg-soft`) recovers 3Di/AA ranking without inflating `rmse_high`** -> the band
   weighting was the cost, not the objective. Deploy the flat/soft weights, keep regression, and Melissa's
   hypothesis is right *about the weights* while colab32 stays right *about the objective*. This is the
   outcome that changes the deployed model.
3. **If the deficit is concentrated in `sp_far` and the high band is untouched** -> that is the mechanism
   confirmed directly: `w_far=0.5` told the encoder not to resolve the region where AA (median 0.20) and
   3Di (median 0.24) keep most of their mass. It also means the overall Spearman on those feeds is
   *dominated by a region the model was deliberately trained to ignore* — which is an argument about the
   metric, and belongs next to the CATH-S20 discussion.
4. **If all four arms are within seed noise on everything except `rmse_high`** -> the honest claim is the
   one already in `WRITEUP.md`: the objective is near-neutral, pooling is the lever, and regression is
   preferred because its readout is aligned and its value fidelity is better. Nothing changes, and the
   3Di question is fully handed to item 1.

**Do not** read a single seed as evidence for any of these. `spearman_high` in particular runs on very few
pairs for AA and 3Di — check `n_high` in the summary table before quoting it.